# Weather Data Extraction

This notebook extracts historical hourly weather data from the Open-Meteo Archive API for six cities in Poland.

The extraction process:
- Defines the cities and their geographical coordinates.
- Specifies the date range and weather variables.
- Configures the API endpoint and timezone.
- Defines a reusable function to retrieve weather data for a single city.
- Implements retry logic to handle temporary connection errors and API rate limits.

In [0]:
# Import libraries used for API requests and execution delays
import requests
import time

# Define cities and their geographical coordinates
CITIES = {
    "Warszawa": {"latitude": 52.2297, "longitude": 21.0122},
    "Kraków": {"latitude": 50.0647, "longitude": 19.9450},
    "Gdańsk": {"latitude": 54.3520, "longitude": 18.6466},
    "Wrocław": {"latitude": 51.1079, "longitude": 17.0385},
    "Poznań": {"latitude": 52.4064, "longitude": 16.9252},
    "Mińsk Mazowiecki": {"latitude": 52.1794, "longitude": 21.5725},
}
# Define the historical data extraction period
START_DATE = "2020-01-01"
END_DATE = "2026-10-29"

# Specify hourly weather variables requested from the API
HOURLY_VARIABLES = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "cloud_cover",
    "shortwave_radiation",
]

# Define the Open-Meteo Historical Weather API endpoint
BASE_URL = "https://archive-api.open-meteo.com/v1/archive"


# Extract hourly weather data for a single city
def extract_city_weather(city, latitude, longitude):
    # API request parameters
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(HOURLY_VARIABLES),
        "timezone": "Europe/Warsaw",
    }
    # Try the API request up to three times in case of connection errors or rate limits
    for attempt in range(3):
        try:
            print(f"Fetching data for {city}, attempt {attempt + 1}/3")

            response = requests.get(url=BASE_URL, params=params, timeout=60)

            # Raise an exception if the API returns an HTTP error
            response.raise_for_status()
            break

        # Handle temporary connection errors
        except requests.exceptions.ConnectionError as error:
            if attempt == 2:
                raise
            wait_seconds = 5 * (2**attempt)

            print(f"Connection error for {city}: {error}")

        # Handle HTTP errors, including API rate limits (HTTP 429)
        except requests.exceptions.HTTPError as error:
            if response.status_code != 429 or attempt == 2:
                raise
            wait_seconds = 5 * (2**attempt)

            print(f"API returned an error {error}")

        # Wait before retrying the request
        for remaining in range(wait_seconds, 0, -1):
            print(f"Retrying for {city} in {remaining} seconds")
            time.sleep(1)
    # Convert the API response into a Python dictionary
    data = response.json()
    hourly = data["hourly"]

    # Create a list to store hourly weather records
    records = []

    # Build one record for each hourly measurement
    for i in range(len(hourly["time"])):
        record = {"city": city, "measurement_time": hourly["time"][i], "latitude": latitude, "longitude": longitude}

        # Add all requested weather variables to the record
        for variable in HOURLY_VARIABLES:
            record[variable] = hourly[variable][i]

        records.append(record)

    # Return all hourly records for the selected city
    return records

# Extract Weather Data for All Cities

This section executes the extraction function for each city defined in the configuration.

The process:
- Iterates through all cities and their geographical coordinates.
- Calls `extract_city_weather()` to retrieve hourly weather data.
- Appends the extracted records to a single list named `all_records`.
- Prints the number of records retrieved for each city.
- Introduces a short delay between API requests to reduce the risk of rate limiting.

At the end of this step, all extracted records are stored together in one collection, ready for validation and further processing.

In [0]:
# Initialize an empty list to store weather records from all cities
all_records = []

# Iterate through all cities defined in the configuration
for city, city_coordinates in CITIES.items():
    # Extract hourly weather data for the current city
    city_records = extract_city_weather(
        city=city,
        latitude=city_coordinates["latitude"],
        longitude=city_coordinates["longitude"],
    )

    # Add the extracted records to the combined list
    all_records.extend(city_records)

    # Print the number of records retrieved for the current city

    print(f"For city {city} retrieved {len(city_records)} records")

    # Wait 10 seconds before requesting data for the next cityeep(10)
    time.sleep(10)

# Display an example of the extracted data
print(f"Example record: {all_records[0]}")

# Display the total number of records collected from all cities
print(f"Total number of records: {len(all_records)}")

# Data Quality Check – Record Counts by City

This section validates the number of extracted records for each city.

The process:
- Iterates through all configured cities.
- Filters the combined `all_records` collection to find records belonging to the current city.
- Counts the matching records.
- Displays the record count for each city.

This check helps verify that data was successfully retrieved for every configured location and that no city has an unexpectedly low number of records.

In [0]:
# Iterate through all configured cities
for city in CITIES:
    # Filter records belonging to the current city
    city_records = [record for record in all_records if record["city"] == city]

    # Count and display the records retrieved for the current city
    print(f"dla miasta {city} ilosc rekordow: {len(city_records)}")

# Data Quality Check – Missing Values

This section examines the completeness of the extracted weather data by identifying missing values.

The validation process:
- Iterates through all requested weather variables.
- Checks each record for missing values (`None`).
- Counts the number of missing values for each variable.
- Displays the results to identify potential gaps in the API response.

The purpose of this check is to assess the quality of the raw extracted data before storing it in the Bronze layer.

**Important:** No records are removed or modified at this stage. Missing values will be further investigated and handled during the Silver layer transformation process.

In [0]:
# Iterate through all requested weather variables
for variable in HOURLY_VARIABLES:

    # Filter records containing missing values fot the current variable
    missing_values = [record for record in all_records if record[variable] == None]
    
    # Display the amount of missing values for current variable
    print(f"For variable {variable} total missing values: {len(missing_values)}")

# Data Quality Check – Duplicate Records

This section checks whether duplicate hourly weather records exist in the extracted dataset.

A combination of `city` and `measurement_time` is treated as a business key that uniquely identifies each hourly weather observation.

The validation process:
- Counts all extracted records.
- Identifies distinct combinations of `city` and `measurement_time`.
- Compares the total number of records with the number of unique combinations.
- Reports whether duplicate records exist.

No records are removed or modified during this validation step.

In [0]:
# Count the total number of records in the extracted dataset
total_count = len(all_records)

# Create a set containing unique city and measurement_time combinations
unique_records = {
    (record["city"], record["measurement_time"]) for record in all_records
}

# Count the number of unique city and measurement_time combinations
unique_count = len(unique_records)

# Calculate the number of duplicate records
total_duplicates = total_count - unique_count

# Display the total number of records
print(f"Total number of records in all_records: {total_count}")
# Display the number of unique records
print(f"Total number of unique records: {unique_count}")
# Display the number of duplicate records
print(f"Total number of duplicate records: {total_duplicates}")

# Validate whether the dataset contains duplicate records
if total_duplicates == 0:
    print("Data quality check passed: no duplicate records found")
else:
    print(f"Data quality check failed: {total_duplicates} duplicates found")

# Data Quality Check – Invalid Values

This section identifies weather measurements that violate basic physical or logical constraints.

The validation process:
- Defines acceptable value ranges for each selected weather variable.
- Checks all extracted records against these rules.
- Counts values that fall outside the expected ranges.
- Reports invalid values for further investigation.

The validation rules are based on physical constraints and the meaning of each weather variable.

No records are removed or modified during this step. Any detected issues will be investigated before data transformation in the Silver layer.


In [0]:
# Define acceptable value ranges for weather variables
VALUE_RANGES = {
    "latitude": (-90, 90),
    "longitude": (-180, 180),
    "temperature_2m": (-100, 100),
    "apparent_temperature": (-100, 100),
    "relative_humidity_2m": (0, 100),
    "precipitation": (0, None),
    "rain": (0, None),
    "snowfall": (0, None),
    "wind_speed_10m": (0, None),
    "wind_direction_10m": (0, 360),
    "wind_gusts_10m": (0, None),
    "cloud_cover": (0, 100),
    "shortwave_radiation": (0, None),
}

# Store records containing invalid values
invalid_records = []

# Iterate through all weather variables and their expected ranges
for variable, (minimum, maximum) in VALUE_RANGES.items():
    # Identify records containing values outside the acceptable range
    invalid_values = [
        record
        for record in all_records
        if (record[variable] < minimum)
        or (maximum is not None and record[variable] > maximum)
    ]

    # Add invalid records to the results collection
    for record in invalid_values:
        invalid_records.append(
            {
                "city": record["city"],
                "measurement_time": record["measurement_time"],
                "variable": variable,
                "value": record[variable],
            }
        )
    # Display the number of invalid values for the current variable
    print(f"{variable}: {len(invalid_values)} invalid values")

# Display examples of detected invalid values
print("Examples of invalid values:")
for record in invalid_records[:10]:
    print(record)

# Convert Python Records into a Spark DataFrame

This step converts the collected weather records from a Python list into a Spark DataFrame.

The DataFrame schema is displayed to verify column names and data types. A preview of the first five records is shown, and the total number of records is checked to ensure that all collected data has been transferred correctly.

The Spark DataFrame will be used for further data processing and storage in Databricks.

In [0]:
# Convert the collected records into a Spark DataFrame
weather_df = spark.createDataFrame(all_records)

# Display the DataFrame structure
weather_df.printSchema()

# Display the first 10 records
display(weather_df.limit(10))

# Display the total number of records
print(f"Total number of records: {weather_df.count()}")

# Create Project Database Schema

This step creates a dedicated schema for the weather data pipeline.

The schema will organize all project tables across the Bronze, Silver, and Gold layers.

In [0]:
# Create a dedicated schema for the weather data pipeline
spark.sql("""CREATE SCHEMA IF NOT EXISTS Workspace.weather_data""")

# Save Raw Weather Data to Bronze Layer

This step saves the extracted raw weather data as a managed Delta table in the dedicated project schema.

The Bronze layer preserves the original data collected from the Open-Meteo API without applying any transformations.

The table will serve as the source for subsequent processing in the Silver layer.

In [0]:
# Define the Bronze table name
BRONZE_TABLE = "Workspace.weather_data.weather_raw"

# Save the raw weather data as a Delta table
weather_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(BRONZE_TABLE)

# Confirm that the table has been created
print(f"Data succesfully saved to {BRONZE_TABLE}")

# Validate Bronze Data

This step verifies that the raw weather data has been successfully stored in the Bronze layer.

The table is queried to check the total number of records and preview the stored data.

In [0]:
# Read the Bronze Delta table
bronze_df = spark.table(BRONZE_TABLE)

# Display the first 10 records
display(bronze_df.limit(10))

# Display the total number of records
print(f"Total number of records: {bronze_df.count()}")